# Paper 1 — TF-IDF + Logistic Regression for Fake News Detection

**Purpose:** Implement a traditional machine-learning baseline and evaluate it on the same type of labeled fake-news data used in the project.

**Pipeline:** preprocessing → TF-IDF → Logistic Regression → evaluation.

> The notebook calculates the result from your dataset; it does not hard-code a paper's accuracy.


In [ ]:
# Install if needed:
# !pip install pandas numpy scikit-learn matplotlib joblib

import re, os, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

RANDOM_STATE=42
TEST_SIZE=0.20
DATA_PATH="fake_news_dataset.csv"   # <-- change this to your CSV


In [ ]:
df=pd.read_csv(DATA_PATH)
print("Shape:",df.shape)
print("Columns:",list(df.columns))
display(df.head())


In [ ]:
# Detect common column names. If detection is wrong, replace these two lines manually.
text_candidates=["text","content","article","news","body","title"]
label_candidates=["label","class","target","category"]
lm={c.lower():c for c in df.columns}
text_col=next((lm[x] for x in text_candidates if x in lm),None)
label_col=next((lm[x] for x in label_candidates if x in lm),None)
if text_col is None or label_col is None:
    raise ValueError("Set text_col and label_col manually after inspecting df.columns.")
print("Text:",text_col," Label:",label_col)


In [ ]:
work=df[[text_col,label_col]].copy()
work.columns=["text","label"]
work=work.dropna()
def clean_text(x):
    x=str(x).lower()
    x=re.sub(r"http\S+|www\.\S+"," ",x)
    x=re.sub(r"[^a-z\s]"," ",x)
    return re.sub(r"\s+"," ",x).strip()
work["clean_text"]=work["text"].map(clean_text)

def encode_labels(s):
    s=s.astype(str).str.strip().str.lower()
    mapping={"fake":0,"false":0,"0":0,"real":1,"true":1,"1":1}
    if set(s.unique()).issubset(mapping): return s.map(mapping).astype(int)
    u=sorted(s.unique())
    if len(u)==2:
        m={u[0]:0,u[1]:1}; print("Automatic label mapping:",m); return s.map(m).astype(int)
    raise ValueError("Binary labels required.")
work["y"]=encode_labels(work["label"])
display(work.head())


## Train/test split

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(
    work["clean_text"],work["y"],test_size=TEST_SIZE,
    random_state=RANDOM_STATE,stratify=work["y"]
)
print(len(X_train),len(X_test))


## TF-IDF feature extraction

In [ ]:
vectorizer=TfidfVectorizer(max_features=5000,ngram_range=(1,2),min_df=2,sublinear_tf=True)
Xtr=vectorizer.fit_transform(X_train)
Xte=vectorizer.transform(X_test)
print("Train matrix:",Xtr.shape," Test matrix:",Xte.shape)


## Logistic Regression

In [ ]:
model=LogisticRegression(max_iter=1000,random_state=RANDOM_STATE)
model.fit(Xtr,y_train)
pred=model.predict(Xte)


## Results

In [ ]:
accuracy=accuracy_score(y_test,pred)
precision=precision_score(y_test,pred,zero_division=0)
recall=recall_score(y_test,pred,zero_division=0)
f1=f1_score(y_test,pred,zero_division=0)
metrics=pd.DataFrame({"Metric":["Accuracy","Precision","Recall","F1-score"],"Score":[accuracy,precision,recall,f1]})
display(metrics)
print(classification_report(y_test,pred,target_names=["Fake","Real"],zero_division=0))


In [ ]:
cm=confusion_matrix(y_test,pred)
fig,ax=plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay(cm,display_labels=["Fake","Real"]).plot(ax=ax)
ax.set_title("Paper 1 — TF-IDF + Logistic Regression")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(5,4))
plt.bar(["TF-IDF + LR"],[accuracy*100])
plt.ylabel("Accuracy (%)"); plt.ylim(0,100)
plt.title("Paper 1 Accuracy")
plt.tight_layout(); plt.show()
print(f"Obtained accuracy: {accuracy*100:.2f}%")


## Example predictions

In [ ]:
examples=[
"The Eiffel Tower is located in Paris.",
"Aliens have opened a shopping mall in New York.",
"The Eiffel Tower is located in London."
]
for text in examples:
    p=model.predict(vectorizer.transform([clean_text(text)]))[0]
    conf=model.predict_proba(vectorizer.transform([clean_text(text)])).max()
    print(text,"->","REAL" if p==1 else "FAKE",f"(confidence {conf:.3f})")


## Comparison with the reference paper

Enter the accuracy reported by your selected reference paper. The notebook computes the difference; it does not claim to reproduce the paper automatically.


In [ ]:
REFERENCE_PAPER_ACCURACY=None  # e.g. 82.0
if REFERENCE_PAPER_ACCURACY is None:
    print(f"Our accuracy: {accuracy*100:.2f}%")
    print("Set REFERENCE_PAPER_ACCURACY to compare.")
else:
    print(f"Our accuracy: {accuracy*100:.2f}%")
    print(f"Reference: {REFERENCE_PAPER_ACCURACY:.2f}%")
    print(f"Difference: {accuracy*100-REFERENCE_PAPER_ACCURACY:+.2f} percentage points")


In [ ]:
os.makedirs("saved_models",exist_ok=True)
joblib.dump(vectorizer,"saved_models/tfidf_vectorizer.pkl")
joblib.dump(model,"saved_models/logistic_regression.pkl")
metrics.to_csv("paper1_metrics.csv",index=False)
print("Saved model, vectorizer and metrics.")
